# Streamflow for the 10 Maribyrnong gauges — BoM exports + Victorian API (self-contained)

Builds one Caravan-ready CSV per gauge (`date`, `streamflow` in mm/day). Run all cells in order.

**Two sources, by design:**
- **7 Melbourne Water gauges** — read from **BoM Water Data Online** exports you download
  yourself (the cleaner, longer, quality-controlled archive; this replaces the Melbourne Water
  API, which carried phantom spikes). Get each one from the BoM Water Data Online portal
  (http://www.bom.gov.au/waterdata/): type the station number (e.g. `230106A`) into the search
  box to zoom the map to the site, click the green icon at the gauge, choose **Water Course
  Discharge** from the left-hand panel, then export/download the series to CSV. Drop the 7
  exports — the raw `.zip` BoM gives you, or an extracted `.csv`, either works — into the `BOM_DIR` folder.
- **3 Victorian Water gauges** (Keilor, Gisborne, Sunbury) — fetched live from the **Hydstra
  API**. Nothing to download; the code pulls them on run.

## What it produces
`caravan_maribyrnong/timeseries/csv/ausvic/<gauge_id>.csv` — `date` (gapless daily) +
`streamflow` (mm/day; empty string for missing). Exactly what Caravan **Part-2** consumes.

## Gauges
| gauge_id | station | source |
|---|---|---|
| ausvic_230119 Deep Ck @ Doggetts Bridge, Lancefield | 230119A | BoM file |
| ausvic_230100 Deep Ck @ Darraweit Guim | 230100A | BoM file |
| ausvic_230107 **Deep Ck @ Konagaderra** | 230107A | BoM file |
| ausvic_230102 Deep Ck d/s Bulla Rd, Bulla | 230102A | BoM file |
| ausvic_230237 Maribyrnong R d/s Jacksons Ck, Keilor Nth | 230237A | BoM file |
| ausvic_230106 Maribyrnong R @ Chifley Dr | 230106A | BoM file |
| ausvic_230211 Bolinda Ck @ Clarkefield | 230211A | BoM file |
| ausvic_230200 Maribyrnong R @ Keilor | 230200 | Victorian Water (Hydstra) |
| ausvic_230206 Jacksons Ck @ Gisborne | 230206 | Victorian Water (Hydstra) |
| ausvic_230202 Jacksons Ck @ Sunbury | 230202 | Victorian Water (Hydstra) |

## BoM aggregation
BoM exports are **sub-daily discharge (m³/s)**. Each is aggregated to a **time-weighted daily
mean** (hourly forward-fill with a 48 h gap tolerance, then the daily mean) and converted
m³/s → ML/day → mm/day with the **GEE Part-1 polygon areas**, so normalisation matches the
`area` in `attributes_other`.

## Data-quality notes (read before submitting)
BoM serves all 7 Melbourne Water gauges as the QC'd `DMQaQc.Merged.AsStored` discharge series,
but the per-reading quality codes vary by gauge (legend in each export's `disclaimer.txt`). This
pipeline KEEPS every gauge (project decision) and does NOT filter on quality code, so these flags
must be documented in `attributes` / README, not silently dropped:

1. **Chifley Dr (`230106`) - tidal; the flood range is quality-A.** This is a flood-event gauge:
   BoM certifies a discharge rating only for high flows here - every reading above ~190 cumecs is
   **quality-A** (the Oct-2022 flood is captured at quality-A, ~494 cumecs daily mean), while
   outside flood events the tidal site is recorded as ~0 (quality-F, no certified rating). So
   ~99% of readings are a zero/uncertified baseline, but the flood peaks - the signal that matters
   for flood forecasting - are real and high-quality. Treat the zeros as "no certified freshwater
   discharge", not measured low flow.
2. **Konagaderra (`230107`) - mixed quality.** ~51% quality-A, ~34% quality-E ("ability to
   represent the parameter is not known"), ~11% quality-F. Usable but lower-confidence.
3. **Sunbury (`230202`) pre-1960** - Hydstra returns data from 1908 but 1908-1959 are ~0.25
   ML/day level artefacts (no rating curve); kept from 1960 only.

(The other four BoM gauges - `230100`, `230119`, `230211`, `230237` - report a 4-digit provider
code, e.g. 1034, not in the A-F legend; each is dominated by a single in-service code and the
daily records validate cleanly. Worth confirming the legend with Melbourne Water/BoM.)

**Retrieval check:** a completeness assert at the very end fails the run if any gauge comes back
below ~95% of its known record (a truncated BoM export, or a Hydstra timeout), so a sparse pull
can't slip through to Part-2.

## Step 0 — Install dependencies

In [ ]:
%pip install -q pandas

## Step 1 — Imports

In [ ]:
import json
import math
import time
import urllib.parse
import urllib.request
import urllib.error
from datetime import date
from pathlib import Path

import pandas as pd

print('imports OK  | pandas', pd.__version__)

## Step 2 — Inline configuration

The 10 gauges with station ID, provider, GEE Part-1 area, and agency-standard name.
Edit `OUT_DIR` to change where the CSVs land.

In [ ]:
# gauge_id, station_id, provider, area_km2 (GEE Part-1), name
GAUGES = [
    ('ausvic_230119', '230119A', 'bom',     219.2154,  'Deep Creek at Doggetts Bridge Lancefield'),
    ('ausvic_230100', '230100A', 'bom',     483.7699,  'Deep Creek at Darraweit Guim'),
    ('ausvic_230107', '230107A', 'bom',     620.7389,  'Deep Creek at Konagaderra'),
    ('ausvic_230102', '230102A', 'bom',     860.3485,  'Deep Creek at Bulla'),
    ('ausvic_230237', '230237A', 'bom',     1279.3986, 'Maribyrnong River at Keilor North'),
    ('ausvic_230106', '230106A', 'bom',     1385.6544, 'Maribyrnong River at Chifley Drive'),
    ('ausvic_230211', '230211A', 'bom',     95.9878,   'Bolinda Creek at Clarkefield'),
    ('ausvic_230200', '230200',  'hydstra', 1306.0112, 'Maribyrnong River at Keilor'),
    ('ausvic_230206', '230206',  'hydstra', 90.9582,   'Jackson Creek at Gisborne'),
    ('ausvic_230202', '230202',  'hydstra', 342.7511,  'Jackson Creek at Sunbury'),
]
GAUGE_DF = pd.DataFrame(GAUGES, columns=['gauge_id', 'station_id', 'provider', 'area_km2', 'name'])

# ---- WHERE YOUR BoM EXPORTS LIVE -------------------------------------------------
# Download each of the 7 Melbourne Water gauges from BoM Water Data Online
# (http://www.bom.gov.au/waterdata/) as "Water Course Discharge", and drop the files
# here. The raw .zip BoM gives you, or an extracted .csv, both work; sub-folders are
# searched and the newest file wins if you happen to have duplicates.
BOM_DIR = Path('C:/Users/leela/FloodHubMaribyrnong/03_streamflow/inputs/bom_exports')
if not any(BOM_DIR.rglob('*230*')):                       # not next to the notebook? walk up and find it
    BOM_DIR = next((p / 'bom_exports' for p in Path.cwd().parents
                    if (p / 'bom_exports').is_dir() and any((p / 'bom_exports').rglob('*230*'))), BOM_DIR)
BOM_DIR.mkdir(exist_ok=True)
BOM_SUBDAILY_GAP_H = 48   # hourly forward-fill tolerance when aggregating sub-daily -> daily
# ---------------------------------------------------------------------------------

# Output: one CSV per gauge, in the structure Caravan Part-2 reads.
OUT_DIR = Path('C:/Users/leela/FloodHubMaribyrnong/03_streamflow/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Per-gauge earliest date to KEEP (inclusive). 230202 pre-1960 = level artefacts.
FETCH_START = {'ausvic_230202': '1960-01-01'}

# Politeness for the 3 live Hydstra calls (BoM gauges are local file reads, no delay).
API_DELAY_S  = 2.0
MAX_RETRIES  = 5     # attempts per Hydstra request before giving up
BACKOFF_BASE = 3.0   # exponential backoff seconds on HTTP 429/503: BASE * 2**attempt

# Minimum data days each gauge must return. For the 7 BoM gauges these are the FULL measured
# pull (Jun-2026) -- a complete read must return at least this many. The BoM record only grows
# forward, so a current/complete export always clears the floor, while a truncated or stale
# export drops below and the run asserts at the end. Counts are pre-QA, so legitimate masking
# can never trip it. The 3 live Hydstra gauges stay at ~95% of the known-good first submission
# (a live API has more run-to-run variance than a local file read).
# BoM full pull (Jun-2026):  230119 8794 | 230100 17350 | 230107 14186 | 230102 25476
#                            230237 9827 | 230106 18113 | 230211 16550
# Known-good Hydstra (first submission): 230200 36490 | 230206 24035 | 230202 24165
MIN_RECORDS = {
    'ausvic_230119':  8794, 'ausvic_230100': 17350, 'ausvic_230107': 14186,
    'ausvic_230102': 25476, 'ausvic_230237':  9827, 'ausvic_230106': 18113,
    'ausvic_230211': 16550, 'ausvic_230200': 34500, 'ausvic_230206': 22800,
    'ausvic_230202': 22900,
}

# QA thresholds
FLATLINE_MIN_RUN   = 5      # >= this many identical non-zero days = stuck sensor
PHYS_CEILING_MM    = 150.0  # daily-mean mm/day ceiling (temperate catchment)
REPEAT_CEILING_MIN = 5      # >= this many identical non-zero days TOTAL (not just
REPEAT_CEILING_MM  = 5.0    #   consecutive), above this mm/day = rating ceiling / infill
FLATLINE_MIN_MM    = 1.0    # only flat-line-mask runs ABOVE this mm/day; low-flow plateaus
                            #   below it are real baseflow on a perennial gauge and are KEPT

print(f'{len(GAUGES)} gauges configured  |  BoM dir -> {BOM_DIR.resolve()}')
print(f'output -> {OUT_DIR.resolve()}')
GAUGE_DF

## Step 2b — HTTP helper (throttle + retry on HTTP 429)

Both APIs can return **HTTP 429 (Too Many Requests)** under load. This helper catches it and
**backs off exponentially** (honoring a numeric `Retry-After` header if the server sends one),
retrying up to `MAX_RETRIES` instead of dropping the request. Same treatment for transient 503 /
network errors. If it still fails after all retries it raises — and the Step 12 assert then flags
the gauge as a failed retrieval.

In [ ]:
def http_get_json(url, headers=None, timeout=60):
    """GET a URL and parse JSON, with exponential backoff + retry on HTTP 429/503
    (rate limiting) and transient network errors. Honors a numeric Retry-After
    header when present. Raises if it still fails after MAX_RETRIES attempts."""
    headers = headers or {}
    for attempt in range(MAX_RETRIES):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode())
        except urllib.error.HTTPError as e:
            if e.code in (429, 503) and attempt < MAX_RETRIES - 1:
                ra = e.headers.get('Retry-After') if e.headers else None
                wait = float(ra) if (ra and str(ra).isdigit()) else BACKOFF_BASE * (2 ** attempt)
                print(f'    [HTTP {e.code} rate-limited] throttling {wait:.0f}s '
                      f'(retry {attempt + 1}/{MAX_RETRIES - 1})')
                time.sleep(wait)
                continue
            raise
        except Exception:
            if attempt < MAX_RETRIES - 1:
                wait = BACKOFF_BASE * (2 ** attempt)
                print(f'    [network error] retry in {wait:.0f}s ({attempt + 1}/{MAX_RETRIES - 1})')
                time.sleep(wait)
                continue
            raise

print('HTTP helper ready  | retry + backoff on HTTP 429/503')

## Step 3 — Read the Melbourne Water gauges from BoM Water Data Online

Each gauge's export is a BoM **"Water Course Discharge"** time series in **m³/s**, sub-daily.
`read_bom_daily_ml()` finds the file for a station in `BOM_DIR` (accepts the raw `.zip` or an
extracted `.csv`, searches sub-folders, newest wins if duplicated), parses the sub-daily series,
forward-fills onto an hourly grid (≤ 48 h gaps), takes the **daily mean**, converts m³/s →
ML/day, and returns `(date, ML/day)` tuples — the same shape the Hydstra fetcher returns, so
everything downstream is identical. The **preflight** below lists which exports were found, so a
missing or misnamed file shows up before you fetch.

In [ ]:
import os, io, glob as _glob, zipfile as _zip

def _find_bom_file(station_id, bom_dir):
    '''Path to the BoM export for this station (newest if several). Accepts the raw
    .zip exports or already-extracted .csv files; searches sub-folders too. Prefers
    an extracted .csv over a .zip, then most-recently-modified.'''
    hits = []
    for ext in ('csv', 'zip'):
        hits += _glob.glob(os.path.join(str(bom_dir), '**', f'*{station_id}*.{ext}'), recursive=True)
    hits = sorted(set(hits), key=lambda p: (p.lower().endswith('.zip'), -os.path.getmtime(p)))
    return hits[0] if hits else None

def _read_bom_text(path):
    '''BoM CSV text, read transparently from a .zip if needed.'''
    if path.lower().endswith('.zip'):
        with _zip.ZipFile(path) as z:
            inner = [n for n in z.namelist() if n.lower().endswith('.csv')]
            if not inner:
                raise FileNotFoundError(f'no .csv inside {os.path.basename(path)}')
            return z.read(inner[0]).decode('utf-8', 'replace')
    with open(path, encoding='utf-8', errors='replace') as fh:
        return fh.read()

def read_bom_daily_ml(station_id, bom_dir):
    '''Read a BoM 'Water Course Discharge' export (sub-daily m3/s) and return a
    time-weighted daily mean as (iso_date, ML/day) tuples -- the same shape
    fetch_vw_flow returns, so the rest of the pipeline is unchanged.

    Aggregation: forward-fill the irregular sub-daily series onto an hourly grid
    (<= BOM_SUBDAILY_GAP_H gap), then the daily mean. Sub-daily tidal backflow is
    kept in the mean (net daily flow); whole days that come out negative are masked
    later by QA. Days with no reading within the gap tolerance are dropped here and
    reappear as gaps (NaN) once build_daily_series rebuilds the gapless index.'''
    path = _find_bom_file(station_id, bom_dir)
    if path is None:
        raise FileNotFoundError(f'no BoM export for {station_id} in {bom_dir} '
                                f'(expected a file whose name contains "{station_id}")')
    text = _read_bom_text(path)
    df = pd.read_csv(io.StringIO(text), comment='#', header=None,
                     names=['ts', 'val', 'q', 'it'], usecols=[0, 1])
    df = df.dropna(subset=['val'])
    if df.empty:
        raise ValueError(f'{station_id}: no data rows parsed from {os.path.basename(path)}')
    idx = pd.to_datetime(df['ts'].astype(str).str[:19], format='%Y-%m-%dT%H:%M:%S', errors='coerce')
    s = pd.Series(pd.to_numeric(df['val'], errors='coerce').values, index=idx)
    s = s[s.index.notna()].dropna().sort_index()
    s = s[~s.index.duplicated(keep='first')]
    grid = pd.date_range(s.index.min().floor('D'), s.index.max().ceil('D'), freq='1h')
    hourly = s.reindex(grid, method='ffill', tolerance=pd.Timedelta(f'{BOM_SUBDAILY_GAP_H}h'))
    daily_m3s = hourly.resample('1D').mean().dropna()   # daily mean cumecs
    daily_ml = daily_m3s * 86.4                          # m3/s -> ML/day
    print(f'    {station_id}: {os.path.basename(path)}  ->  {len(daily_ml):,} days  '
          f'{daily_ml.index.min().date()}..{daily_ml.index.max().date()}')
    return [(d.strftime('%Y-%m-%d'), float(v)) for d, v in daily_ml.items()]

# preflight: which BoM exports were located (a missing/misnamed file is obvious now)
print('BoM exports located in', BOM_DIR.resolve())
for _sid in [g['station_id'] for _, g in GAUGE_DF.iterrows() if g['provider'] == 'bom']:
    _p = _find_bom_file(_sid, BOM_DIR)
    print(('  [ok]      ' + _sid + '  ' + os.path.basename(_p)) if _p
          else ('  [MISSING] ' + _sid + '  <- put its BoM .zip/.csv in ' + str(BOM_DIR)))
print('BoM reader ready  | sub-daily m3/s -> daily mean ML/day')

## Step 4 — Victorian Water / Hydstra API

`GET https://data.water.vic.gov.au/cgi/webservice.exe` · `get_ts_traces`, var `141.00` (ML/day),
datasource `PUBLISH` → `return.traces[0].trace` = list of `{t, v, q}`. Quality 255 (bad) and
negatives dropped.

In [ ]:
HYDSTRA_BASE = 'https://data.water.vic.gov.au/cgi/webservice.exe'

def fetch_vw_flow(station_id):
    """All daily mean discharge (var 141.00 = ML/day) for a Victorian Water (Hydstra)
    station, as (iso_date, ml) tuples. Drops quality flag 255 (bad) and negatives."""
    params = {
        'function':   'get_ts_traces',
        'version':    '2',
        'site_list':  station_id,
        'datasource': 'PUBLISH',
        'varfrom':    '100.00',
        'varto':      '141.00',
        'start_time': '19000101000000',
        'end_time':   date.today().strftime('%Y%m%d235959'),
        'interval':   'day',
        'multiplier': '1',
        'data_type':  'mean',
    }
    url = f'{HYDSTRA_BASE}?{urllib.parse.urlencode(params)}'
    data = http_get_json(url, timeout=60)
    traces = data.get('return', {}).get('traces', [])
    if not traces:
        print(f'    [warn] no traces for {station_id}')
        return []
    rows = []
    for pt in traces[0].get('trace', []):
        if pt.get('q') == 255 or pt.get('v') in ('', None):
            continue
        raw = float(pt['v'])
        if raw < 0:
            continue
        ts = str(pt['t'])
        if len(ts) < 8:
            continue
        rows.append((f'{ts[:4]}-{ts[4:6]}-{ts[6:8]}', raw))
    return rows

print('Victorian Water (Hydstra) fetcher ready.')

## Step 5 — Convert to mm/day on a gapless daily index

`mm/day = ML/day ÷ area_km2`. De-duplicate on date (keep first), then reindex to a complete
daily spine so Part-2's no-gaps check passes. Missing days become NaN (written as empty string).

In [ ]:
def build_daily_series(rows_ml, area_km2, fetch_start=None):
    """(iso_date, ml/day) tuples -> gapless daily pandas Series in mm/day."""
    if not rows_ml:
        return pd.Series(dtype=float, name='streamflow')
    df = (pd.DataFrame(rows_ml, columns=['date', 'flow_ml'])
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .sort_values('date')
            .drop_duplicates('date', keep='first')
            .set_index('date'))
    if fetch_start:
        df = df.loc[fetch_start:]
    if df.empty:
        return pd.Series(dtype=float, name='streamflow')
    df['streamflow'] = df['flow_ml'] / area_km2
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='D')
    s = df['streamflow'].reindex(full_idx)
    s.index.name = 'date'
    s.name = 'streamflow'
    return s

print('Converter ready.')

## Step 6 — Fetch every gauge

In [ ]:
results = {}     # gauge_id -> Series (mm/day, gapless daily)
DATA_DAYS = {}   # gauge_id -> data days obtained (pre-QA, post-clip) for the retrieval check

for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']; sid = g['station_id']; src = g['provider']; area = g['area_km2']
    print(f'  {gid} ({sid}, {src}) ...', flush=True)
    try:
        rows = read_bom_daily_ml(sid, BOM_DIR) if src == 'bom' else fetch_vw_flow(sid)
    except Exception as e:
        print(f'    ERROR: {e}')
        results[gid] = pd.Series(dtype=float, name='streamflow')
        DATA_DAYS[gid] = 0
        if src == 'hydstra':
            time.sleep(API_DELAY_S)
        continue
    s = build_daily_series(rows, area, fetch_start=FETCH_START.get(gid))
    results[gid] = s
    DATA_DAYS[gid] = int(s.notna().sum())
    if len(s):
        miss = 100 * s.isna().sum() / len(s)
        d0, d1 = s.index.min().date(), s.index.max().date()
        print(f'    -> {int(s.notna().sum()):>6} valid / {len(s)} days | {miss:4.1f}% missing | {d0}..{d1}')
    else:
        print('    -> no data')
    if src == 'hydstra':
        time.sleep(API_DELAY_S)

## Step 7 — QA pass (flag, never modify)

**Diagnostic only.** Counts and reports, per gauge, any negatives, daily means > 150 mm/day,
flat-lines (>= 5 identical values > 1 mm/day) and global repeat-ceilings -- to **alert you** -- but
does **not** change any values. The streamflow is left exactly as the source (BoM / Hydstra)
provided it, per project decision; investigate any flagged gauge yourself rather than auto-masking.

In [ ]:
def qa_clean(series):
    '''DIAGNOSTIC ONLY -- flags negatives, >150 mm/day, flat-lines (>=5 identical >1mm) and
    global repeat-ceilings and reports the counts, but does NOT modify the series. Values are
    left exactly as the source (BoM / Hydstra) provided them, per project decision. Returns the
    unchanged series + a report so the developer can investigate any flagged gauge.'''
    s = series
    rep = {'flatline': 0, 'ceiling': 0, 'negative': 0, 'repeat': 0, 'examples': []}

    rep['negative'] = int((s < 0).sum())

    hi = s > PHYS_CEILING_MM
    rep['ceiling'] = int(hi.sum())
    for dt, v in s[hi].items():
        rep['examples'].append(f'ceiling {dt.date()} {v:.1f} mm/day')

    vals = s.values
    n = len(vals); i = 0
    while i < n:
        v = vals[i]
        if v is not None and not (isinstance(v, float) and math.isnan(v)) and v != 0:
            j = i
            while j + 1 < n and vals[j + 1] == v:
                j += 1
            run = j - i + 1
            if run >= FLATLINE_MIN_RUN and v > FLATLINE_MIN_MM:
                rep['flatline'] += run
                rep['examples'].append(f'flatline {s.index[i].date()}..{s.index[j].date()} ({run}d @ {v:.2f} mm/day)')
            i = j + 1
        else:
            i += 1

    vc = s[s > REPEAT_CEILING_MM].value_counts()
    stuck = vc[vc >= REPEAT_CEILING_MIN].index
    if len(stuck):
        rep['repeat'] = int(s.isin(stuck).sum())
        for val in stuck:
            rep['examples'].append(f'repeat-ceiling {val:.4f} mm/day x{int((s == val).sum())} days')

    return s, rep

print('QA pass (flat-line >=5d, >150 mm/day, negatives) -- DIAGNOSTIC ONLY, values not modified')
print('=' * 70)
for gid in list(results):
    results[gid], rep = qa_clean(results[gid])
    fl, ce, ng, rp = rep['flatline'], rep['ceiling'], rep['negative'], rep['repeat']
    tot = fl + ce + ng + rp
    if tot:
        print(f'{gid}: {tot} day(s) (flatline={fl}, ceiling={ce}, neg={ng}, repeat={rp})')
        for ex in rep['examples'][:8]:
            print(f'    - {ex}')
    else:
        print(f'{gid}: clean')

## Step 8 — Save one CSV per gauge

`date,streamflow` — mm/day to 4 dp, empty string for missing. Part-2 reads these with
`pd.to_numeric(..., errors="coerce")`.

In [ ]:
saved = []
for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']
    s = results.get(gid)
    if s is None or len(s) == 0:
        print(f'  [skip] {gid} - no data')
        continue
    out = s.round(4).to_frame()
    out['streamflow'] = out['streamflow'].map(lambda x: '' if pd.isna(x) else f'{x:.4f}')
    p = OUT_DIR / f'{gid}.csv'
    out.to_csv(p)
    saved.append(gid)
    print(f'  saved {p}  ({len(out)} rows)')
print(f'\n{len(saved)} / {len(GAUGE_DF)} gauges saved.')

## Step 9 — Summary

In [ ]:
hdr = (f'{"gauge_id":<16}{"station":>9}{"provider":>11}{"area km2":>10}'
       f'{"valid":>8}{"missing":>9}  period')
print(hdr)
print('-' * len(hdr))
for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']; sid = g['station_id']; src = g['provider']; ar = g['area_km2']
    s = results.get(gid, pd.Series(dtype=float))
    if len(s) == 0:
        print(f'{gid:<16}{sid:>9}{src:>11}{ar:>10.1f}{"NO DATA":>8}')
        continue
    miss = 100 * s.isna().sum() / len(s)
    period = f'{s.index.min().date()} .. {s.index.max().date()}'
    print(f'{gid:<16}{sid:>9}{src:>11}{ar:>10.1f}{int(s.notna().sum()):>8}{miss:>8.1f}%  {period}')
print()
print('Notes: 7 Melbourne Water gauges are from BoM Water Data Online exports (sub-daily m3/s);')
print('       3 (Keilor/Gisborne/Sunbury) are live from the Victorian Water (Hydstra) API.')
print('       230106 (Chifley Dr) is tidal ')
print('       230102 (Bulla) carries the long record (back to 1955).')

## Step 10 — Sanity check: top-10 flows per gauge

Cross-check the biggest events against MMBW 1986 Table 7.1 / Jacobs 2023, e.g. Keilor (230200):
1974 ~714, 2022 ~768, 1916 642, 1993 ~650 cumecs; Sunbury (230202): Feb-1951 ~227, Oct-1983 ~225.

In [ ]:
AREA = {g['gauge_id']: g['area_km2'] for _, g in GAUGE_DF.iterrows()}
print('Top-10 daily flows per gauge')
print('=' * 72)
for gid, s in results.items():
    if len(s) == 0 or s.notna().sum() == 0:
        print(f'\n{gid}: NO DATA')
        continue
    area = AREA[gid]
    print(f'\n{gid}  (area {area:.1f} km2)  {int(s.notna().sum()):,} valid days')
    print(f'  {"date":<12}{"mm/day":>9}{"ML/day":>11}{"cumecs":>10}')
    for dt, mm in s.dropna().nlargest(10).items():
        ml = mm * area
        m3s = ml / 86.4
        print(f'  {str(dt.date()):<12}{mm:>9.2f}{ml:>11.1f}{m3s:>10.1f}')

## Step 11 — Final verification

Re-reads the saved CSVs from disk and asserts each is Caravan-ready: exactly `date,streamflow`,
a gapless daily index, and no negatives / no > 150 mm/day surviving QA.

In [ ]:
CHK, CROSS = '✓', '✗'
checks = {}
n_ok = 0
for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']
    p = OUT_DIR / f'{gid}.csv'
    if not p.is_file():
        checks[gid] = 'MISSING FILE'
        continue
    df = pd.read_csv(p)
    cols_ok = list(df.columns) == ['date', 'streamflow']
    dts = pd.to_datetime(df['date'])
    gapless = (len(dts) > 0
               and (dts.max() - dts.min()).days + 1 == len(dts)
               and dts.is_monotonic_increasing
               and not dts.duplicated().any())
    vals = pd.to_numeric(df['streamflow'], errors='coerce')
    n_neg = int((vals < 0).sum())
    n_hi = int((vals > PHYS_CEILING_MM).sum())
    ok = cols_ok and gapless          # structural gate only -- data-quality flags below are FYI, not failures
    n_ok += int(ok)
    flags = []
    if n_neg: flags.append(f'{n_neg} neg')
    if n_hi:  flags.append(f'{n_hi} >150mm')
    fyi = (' | FYI: ' + ', '.join(flags)) if flags else ''
    checks[gid] = ('OK' if ok else f'cols={cols_ok} gapless={gapless}') + fyi

for gid, st in checks.items():
    print(f'  {CHK if st.startswith("OK") else CROSS} {gid}: {st}')
assert n_ok == len(GAUGE_DF), f'{len(GAUGE_DF) - n_ok} gauge(s) failed STRUCTURAL checks (cols / gapless)'
print(f'\nAll {n_ok}/{len(GAUGE_DF)} gauges pass structural checks (date+streamflow, gapless daily).')
print('Data-quality flags above are FYI only -- values are left exactly as the source provided.')
print('Ready for Caravan Part-2.')

## Step 12 — Retrieval-completeness check (fails on a partial/truncated pull)

Asserts each gauge returned at least `MIN_RECORDS` **data days** — ~95% of its known record (the
minimum the BoM exports and the Hydstra API have actually delivered). A truncated BoM file in
`BOM_DIR`, or a Hydstra timeout, lands below the floor and fails the run here, so a sparse pull
can't reach Part-2. Counts are pre-QA, so legitimate QA masking can't trip it.

In [ ]:
CHK, CROSS = '✓', '✗'
print('Data days obtained vs minimum (the source is known to deliver these):')
short = []
for _, g in GAUGE_DF.iterrows():
    gid = g['gauge_id']
    n = DATA_DAYS.get(gid, 0)
    lo = MIN_RECORDS.get(gid, 0)
    ok = n >= lo
    if not ok:
        short.append(gid)
    print(f'  {CHK if ok else CROSS} {gid}: {n:>7,} days  (min {lo:,})')
assert not short, (f'INCOMPLETE RETRIEVAL for {short} - re-check. A BoM gauge below its floor '
                   f'usually means a truncated or stale export in BOM_DIR; a Hydstra gauge means '
                   f'an API timeout (raise API_DELAY_S and re-run).')
print('\nAll gauges returned their full known record.')

## Next step

Run **Caravan Part-2** (local postprocessing) to merge these `caravan_maribyrnong/timeseries/csv/ausvic/*.csv`
streamflow files with the ERA5-Land batch exports from Part-1. Part-2 is also where
`attributes_other` is written — make sure its gauge-name source has **"Deep Creek at Konagaderra"** for 230107.